In [12]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV,
    RandomizedSearchCV
)

from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from scipy.stats import randint

In [13]:
df = pd.read_csv("bank_marketing.csv")
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,32,admin.,married,secondary,no,11434,no,no,cellular,14,jan,152,3,259,10,unknown,no
1,87,student,single,secondary,no,-1149,no,yes,cellular,9,mar,911,11,195,2,failure,no
2,62,housemaid,single,primary,no,14361,no,no,cellular,27,nov,2563,12,-1,0,unknown,yes
3,23,unemployed,single,secondary,no,28512,no,no,cellular,27,jun,696,12,-1,0,unknown,yes
4,27,housemaid,married,secondary,no,58589,yes,no,cellular,18,apr,2834,11,-1,0,unknown,yes


In [14]:
print(df.shape)
#print columns name
print(df.columns.tolist)

(20000, 17)
<bound method IndexOpsMixin.tolist of Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'y'],
      dtype='object')>


In [15]:
df.columns = df.columns.str.strip()
print(df.columns)

Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'y'],
      dtype='object')


In [16]:
target = "y"
print(df.isnull().sum())

age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64


In [17]:
df.dtypes

age           int64
job          object
marital      object
education    object
default      object
balance       int64
housing      object
loan         object
contact      object
day           int64
month        object
duration      int64
campaign      int64
pdays         int64
previous      int64
poutcome     object
y            object
dtype: object

In [18]:
print(df[target].value_counts())
#label encoding
encoder = LabelEncoder()
for col in df.select_dtypes(include="object").columns:
    df[col] = encoder.fit_transform(df[col])

y
yes    13307
no      6693
Name: count, dtype: int64


In [19]:
#feature and target
X = df.drop(columns=[target])
y = df[target]
print("feature shape = ",X.shape)
print("Target Shape =",y.shape)

feature shape =  (20000, 16)
Target Shape = (20000,)


In [20]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = 0.20, random_state = 42)
print("Training ",X_train.shape)
print("Testing ",X_test.shape)


Training  (16000, 16)
Testing  (4000, 16)


In [22]:
model = DecisionTreeClassifier(random_state=42)

model.fit(X_train, y_train)

prediction = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, prediction), 4))

print(classification_report(y_test, prediction))

print("Confusion Matrix:\n", confusion_matrix(y_test, prediction))

Accuracy: 0.7308
              precision    recall  f1-score   support

           0       0.58      0.64      0.61      1308
           1       0.82      0.77      0.79      2692

    accuracy                           0.73      4000
   macro avg       0.70      0.71      0.70      4000
weighted avg       0.74      0.73      0.73      4000

Confusion Matrix:
 [[ 841  467]
 [ 610 2082]]


In [24]:
#cross validations 
cv = StratifiedKFold(
    n_splits = 5,
    shuffle=True,
    random_state=42
)
scores = cross_val_score(model,X,y,cv=cv,scoring="accuracy")
print(scores)
print("Average accuracy",round(scores.mean(),4))
print ("Standard Deviation",round(scores.std(),4))

[0.73175 0.72075 0.7305  0.7225  0.729  ]
Average accuracy 0.7269
Standard Deviation 0.0044


In [31]:
param_grid = {
    "criterion": ["gini", "entropy"],
    "max_depth": [3, 5, 7, 10, 15, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 6]
}

grid = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best Cross Validation Accuracy:", round(grid.best_score_, 4))
grid_prediction = grid.predict(X_test)
print("Grid Search CV Test Accuracy:", round(accuracy_score(y_test, grid_prediction), 4))


Best Parameters: {'criterion': 'entropy', 'max_depth': 7, 'min_samples_leaf': 1, 'min_samples_split': 10}
Best Cross Validation Accuracy: 0.8262
Grid Search CV Test Accuracy: 0.8427


In [32]:
param_dist = {
    "criterion": ["gini", "entropy"],
    "max_depth": randint(3, 20),
    "min_samples_split": randint(2, 15),
    "min_samples_leaf": randint(2, 8)
}

random = RandomizedSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)

random.fit(X_train, y_train)

print("Best Parameters:", random.best_params_)
print("Best cv score :", round(random.best_score_, 4))
random_prediction = random.predict(X_test)
print("Random Search CV Test Accuracy:", round(accuracy_score(y_test, random_prediction), 4))



Best Parameters: {'criterion': 'entropy', 'max_depth': 8, 'min_samples_leaf': 7, 'min_samples_split': 11}
Best cv score : 0.8262
Random Search CV Test Accuracy: 0.8387
